In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s4e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s4e10/train.csv
/kaggle/input/competitions/playground-series-s4e10/test.csv


In [2]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s4e10/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s4e10/test.csv')

In [3]:
print(train.shape)
print(train.info())
print(train.head())

(58645, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58645 entries, 0 to 58644
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          58645 non-null  int64  
 1   person_age                  58645 non-null  int64  
 2   person_income               58645 non-null  int64  
 3   person_home_ownership       58645 non-null  object 
 4   person_emp_length           58645 non-null  float64
 5   loan_intent                 58645 non-null  object 
 6   loan_grade                  58645 non-null  object 
 7   loan_amnt                   58645 non-null  int64  
 8   loan_int_rate               58645 non-null  float64
 9   loan_percent_income         58645 non-null  float64
 10  cb_person_default_on_file   58645 non-null  object 
 11  cb_person_cred_hist_length  58645 non-null  int64  
 12  loan_status                 58645 non-null  int64  
dtypes: float64(3), int6

In [4]:
train.isnull().sum()

id                            0
person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
loan_status                   0
dtype: int64

In [5]:
X = train.drop('loan_status',axis=1)
y = train["loan_status"]

In [6]:
for df in [train, test]:
    df["income_per_age"] = df["person_income"] / (df["person_age"] + 1)
    df["loan_to_income"] = df["loan_amnt"] / (df["person_income"] + 1)
    df["interest_amount"] = df["loan_amnt"] * df["loan_int_rate"] / 100
    df["cred_hist_age_ratio"] = (
        df["cb_person_cred_hist_length"] /
        (df["person_age"] + 1)
    )

In [7]:
cat_features = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade",
    "cb_person_default_on_file"
]

In [8]:
X_test = test.copy()
test_preds = np.zeros(len(X_test))

In [9]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(X))

for train_idx, valid_idx in folds.split(X, y):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        eval_metric='AUC',
        verbose=False
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        cat_features=cat_features
    )
    test_preds += model.predict_proba(X_test)[:, 1] / folds.n_splits

    oof[valid_idx] = model.predict_proba(X_valid)[:, 1]

score = roc_auc_score(y, oof)

print("CV AUC:", score)

CV AUC: 0.951142577356471


In [10]:
print(test_preds[:20])

[0.99832259 0.02241116 0.45574489 0.01141333 0.06770351 0.95152137
 0.00211278 0.00685639 0.25016897 0.02064313 0.03292324 0.10298545
 0.04566063 0.00640251 0.02370614 0.1054565  0.01075683 0.00251942
 0.03454958 0.00811685]


In [11]:
submission = pd.DataFrame({
    "id": test["id"],
    "loan_status": test_preds
})

submission.to_csv("submission.csv", index=False)

In [12]:
submission.head()

,id,loan_status
0,58645,0.998323
1,58646,0.022411
2,58647,0.455745
3,58648,0.011413
4,58649,0.067704
